# Ho Chi Minh City - PM2.5 Air Quality Forecasting

## Project Overview
This is a **univariate time series forecasting problem**. The target variable is `PM2.5` (fine particulate matter, µg/m³) measured hourly at Station 1 in HCMC (Ho Chi Minh City). Raw sensor data is ingested from CSV into a local **MongoDB** database, then queried, wrangled, and modelled using three progressively sophisticated approaches:
- **Linear Regression with a lag feature** - bridges tabular ML intuition into time series thinking
- **AutoRegression (AR)** - uses multiple past hours simultaneously; lag order selected via PACF analysis
- **ARMA (AutoRegressive Moving Average)** - adds a moving average component; best (p,q) found via grid search

All models are evaluated using **walk-forward validation**, the time series equivalent of cross-validation.

**Target variable:** `PM2.5` (µg/m³)  
**Dataset:** HealthyAir - Outdoor Air Quality in Ho Chi Minh City, Vietnam (Mendeley Data, CC BY 4.0)  
**Source:** https://data.mendeley.com/datasets/pk6tzrjks8/1  
**Citation:** Rakholia et al. (2022). DOI: 10.17632/pk6tzrjks8.1

---
## 1: Setup & Data Ingestion into MongoDB

In [1]:
from pprint import PrettyPrinter
import pandas as pd
from pymongo import MongoClient

pp = PrettyPrinter(indent=2)

In [2]:
# Connect to local MongoDB
client = MongoClient(host='localhost', port=27017)
print('Connected. Databases:', client.list_database_names())

Connected. Databases: ['admin', 'config', 'local']


In [3]:
# Connect to our database and collection
db = client['air-quality']
hcmc = db['hcmc']

# Ingest CSV into MongoDB
if hcmc.count_documents({}) == 0:
    print('Collection empty. Loading CSV into MongoDB...')
    raw = pd.read_csv('data/AirQuality_hcmc.csv')
    print('CSV shape:', raw.shape)
    print('Columns:', raw.columns.tolist())
    hcmc.insert_many(raw.to_dict(orient='records'))
    print(f'Inserted {hcmc.count_documents({}):,} documents.')
else:
    print(f'Already loaded: {hcmc.count_documents({}):,} documents.')

Collection empty. Loading CSV into MongoDB...
CSV shape: (52548, 10)
Columns: ['date', 'Station_No', 'TSP', 'PM2.5', 'O3', 'CO', 'NO2', 'SO2', 'Temperature', 'Humidity']
Inserted 52,548 documents.


In [4]:
# Inspect one document to understand the structure
pp.pprint(hcmc.find_one({}))

{ 'CO': 1330.451429,
  'Humidity': 63.18809524,
  'NO2': 112.7407619,
  'O3': 55.43138095,
  'PM2.5': 15.6047619,
  'SO2': 393.0,
  'Station_No': 1,
  'TSP': 32.93571429,
  'Temperature': 28.36190476,
  '_id': ObjectId('69c663563bc900a6b9e2b94b'),
  'date': '23-02-2021 21:00'}


In [5]:
# How many stations and readings per station?
print('Unique stations:', hcmc.distinct('Station_No'))
result = hcmc.aggregate([
    {'$group': {'_id': '$Station_No', 'count': {'$count': {}}}},
    {'$sort': {'_id': 1}}
])
pp.pprint(list(result))

Unique stations: [1, 2, 3, 4, 5, 6]
[ {'_id': 1, 'count': 7892},
  {'_id': 2, 'count': 9357},
  {'_id': 3, 'count': 8418},
  {'_id': 4, 'count': 9951},
  {'_id': 5, 'count': 7431},
  {'_id': 6, 'count': 9499}]


In [6]:
# Preview PM2.5 readings from Station 1 (traffic zone)
result = hcmc.find(
    {'Station_No': 1},
    projection={'date': 1, 'PM2.5': 1, 'Temperature': 1, 'Humidity': 1, '_id': 0}
).limit(5)
pp.pprint(list(result))

[ { 'Humidity': 63.18809524,
    'Temperature': 28.36190476,
    'date': '23-02-2021 21:00'},
  { 'Humidity': 63.77352941,
    'Temperature': 28.32058824,
    'date': '23-02-2021 22:00'},
  {'Humidity': 64.205, 'Temperature': 28.33666667, 'date': '23-02-2021 23:00'},
  {'Humidity': 64.735, 'Temperature': 28.305, 'date': '24-02-2021 00:00'},
  {'Humidity': 65.18833333, 'Temperature': 28.3, 'date': '24-02-2021 01:00'}]
